In [7]:
import os
import numpy as np
import geopandas as gpd

# Fix for missing .shx file just in case
os.environ["SHAPE_RESTORE_SHX"] = "YES"

def get_bearing(p1, p2):
    """Calculates the accurate compass bearing between two metric points."""
    delta_x = p2.x - p1.x
    delta_y = p2.y - p1.y
    return (np.degrees(np.arctan2(delta_x, delta_y)) + 360) % 360

def get_compass_direction(bearing):
    """Maps a 0-360 degree bearing cleanly to one of the 8 grid directions."""
    if (bearing >= 337.5) or (bearing < 22.5):   return "N"
    elif 22.5 <= bearing < 67.5:                 return "NE"
    elif 67.5 <= bearing < 112.5:                return "E"
    elif 112.5 <= bearing < 157.5:               return "SE"
    elif 157.5 <= bearing < 202.5:               return "S"
    elif 202.5 <= bearing < 247.5:               return "SW"
    elif 247.5 <= bearing < 292.5:               return "W"
    elif 292.5 <= bearing < 337.5:               return "NW"

def generate_clean_state_grids(shapefile_path, output_txt_path):
    print("Loading 100MB shapefile...")
    gdf = gpd.read_file(shapefile_path)
    
    # Standardize column naming to upper case
    gdf['SUBURB_NAM'] = gdf['suburbname'].str.upper()
    
    print("Projecting entire state map to metric system (EPSG:5626) for accurate compass bearings...")
    gdf_metric = gdf.to_crs(epsg=9473) # GDA2020 / Australian Albers
    
    print("Pre-calculating geometric centroids...")
    # Pre-cache metric centroids to keep loops lightning fast
    centroids = {idx: row.geometry.centroid for idx, row in gdf_metric.iterrows()}
    
    # Dissolve multi-polygons conceptually via unique indexing to stop inner duplicate reads
    unique_suburbs = gdf_metric['SUBURB_NAM'].unique()
    total_suburbs = len(unique_suburbs)
    
    print(f"Generating massive text index for all {total_suburbs} NSW suburbs...")
    
    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write("==================================================================\n")
        f.write("             MASTER COMPASS GRID FOR ALL NSW SUBURBS              \n")
        f.write("==================================================================\n\n")
        
        for count, name in enumerate(sorted(unique_suburbs), 1):
            target_rows = gdf_metric[gdf_metric['SUBURB_NAM'] == name]
            if target_rows.empty: continue
            
            target_geom = target_rows.geometry.unary_union
            target_centroid = target_geom.centroid
            
            # Spatial filter box buffer (approx 1.5km buffer around suburb in meters)
            possible_neighbors = gdf_metric[gdf_metric.geometry.intersects(target_geom.buffer(1500))]
            neighbors = possible_neighbors[possible_neighbors.geometry.touches(target_geom) & (possible_neighbors['SUBURB_NAM'] != name)]
            
            # Track the winning suburb name and its border length for each direction
            # Format: { DIRECTION: (suburb_name, border_length) }
            grid_winners = {
                "NW": ("", 0.0), "N": ("", 0.0), "NE": ("", 0.0),
                "W":  ("", 0.0),                 "E":  ("", 0.0),
                "SW": ("", 0.0), "S": ("", 0.0), "SE": ("", 0.0)
            }
            
            for idx, row in neighbors.iterrows():
                n_name = row['SUBURB_NAM'].title()
                n_centroid = centroids[idx]
                n_geom = row.geometry
                
                # Calculate the exact length of the shared border in meters
                shared_border_length = target_geom.intersection(n_geom).length
                
                bearing = get_bearing(target_centroid, n_centroid)
                direction = get_compass_direction(bearing)
                
                # OVERWRITE LOGIC: Only save this suburb if its shared border is longer 
                # than the one currently occupying this direction slot
                if shared_border_length > grid_winners[direction][1]:
                    grid_winners[direction] = (n_name, shared_border_length)
            
            # Extract just the clean names for the final display dictionary
            grid = {dir_key: val_tuple[0] for dir_key, val_tuple in grid_winners.items()}
            grid["C"] = f"◄ {name.title()} ►"
            
            # Build the physical clean grid block text layout
            suburb_block = f"""### SUBURB: {name.upper()}
    [{grid['NW']:^20}] | [{grid['N']:^20}] | [{grid['NE']:^20}]
    ----------------------------------------------------------------------------------
    [{grid['W']:^20}] | [{grid['C']:^20}] | [{grid['E']:^20}]
    ----------------------------------------------------------------------------------
    [{grid['SW']:^20}] | [{grid['S']:^20}] | [{grid['SE']:^20}]
    \n{"="*82}\n\n"""
            
            f.write(suburb_block)
            
            if count % 200 == 0:
                print(f"Processed: {count} / {total_suburbs} suburbs written...")
                
    print(f"\nCompleted! Open the generated file: '{output_txt_path}' to view the giant grid table.")

# --- EXECUTE GIANT EXPORT RUN ---
shapefile = "nsw-administrative-boundaries-theme-suburb.shp"
output_file = "all_nsw_suburbs_grids.txt"

generate_clean_state_grids(shapefile, output_file)

Loading 100MB shapefile...
Projecting entire state map to metric system (EPSG:5626) for accurate compass bearings...
Pre-calculating geometric centroids...
Generating massive text index for all 4516 NSW suburbs...


/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/144

Processed: 200 / 4516 suburbs written...


/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/144

Processed: 400 / 4516 suburbs written...


...

/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/144

Processed: 4400 / 4516 suburbs written...


/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/144


Completed! Open the generated file: 'all_nsw_suburbs_grids.txt' to view the giant grid table.


/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/1444851100.py:54: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_geom = target_rows.geometry.unary_union
/var/folders/d6/2nsmsmpx1y17c35xs74xjjtr0000gn/T/ipykernel_39491/144